In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high_small/checkpoints/post_cell_15_try_1.pickle

In [4]:
%%RecordEvent
%%time
### cell 16 ###

# filter to June
df = flights_df[flights_df['month'] == 6]

# use the original .agg with numpy mean to match the exact output
# (avoids subtle floating-point differences from the GPU .mean)
df = df.groupby('carrier', as_index=False).agg({
    'arr_delay': np.mean,
    'dep_delay': np.mean
})

# vectorized total_delay calculation
df['total_delay'] = df['arr_delay'] + df['dep_delay']

# find and print all carrier(s) with the minimum total_delay
min_total = df['total_delay'].min()
min_df = df[df['total_delay'] == min_total]
for c, t in zip(min_df['carrier'], min_df['total_delay']):
    print(c, t)

# return the DataFrame
df

HA 3.3
CPU times: user 49.8 ms, sys: 28.4 ms, total: 78.3 ms
Wall time: 78 ms


/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/fast_slow_proxy.py:28: FutureWarning: The provided callable <function mean at 0x7fb4dc65fd00> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  return fn(*args, **kwargs)
/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/fast_slow_proxy.py:28: FutureWarning: The provided callable <function mean at 0x7fb4dc65fd00> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  return fn(*args, **kwargs)


,carrier,arr_delay,dep_delay,total_delay
0,9E,22.511905,28.952978,51.464883
1,AA,6.481178,14.627778,21.108956
2,AS,-3.750000,13.083333,9.333333
3,B6,18.541319,20.392170,38.933488
4,DL,13.261829,18.735941,31.997770
5,EV,21.267975,25.496834,46.764809
6,F9,26.636364,29.436364,56.072727
7,FL,41.966805,38.806584,80.773389
8,HA,1.833333,1.466667,3.300000
9,MQ,23.143722,20.842342,43.986064


In [ ]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high_small/checkpoints/post_cell_16_try_2.pickle